<a href="https://colab.research.google.com/github/ronykris/capstone-group10/blob/feat-detection/yolo11_foodDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 902.2/902.2 kB 32.4 MB/s eta 0:00:00


In [4]:
import torch
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
!pip install opencv-python pillow tqdm pyyaml numpy

In [ ]:
!unzip /content/YoloDatasetUpdated.zip

Streaming output truncated to the last 5000 lines.
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0092.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0177.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0462.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0403.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0249.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0137.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0100.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0269.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0333.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0270.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0195.jpg  
  inflating: content/YoloDatasetUpdated/synthetic/images/synthetic_0471.jpg  
  inflating: 

In [1]:
!git clone https://github.com/binayakpokhrel/datasets.git newdataset

Cloning into 'newdataset'...
remote: Enumerating objects: 1063, done.
remote: Counting objects: 100% (7/7), done.
remote: Total 1063 (delta 6), reused 6 (delta 6), pack-reused 1056 (from 1)
Receiving objects: 100% (1063/1063), 2.39 GiB | 35.36 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Updating files: 100% (965/965), done.


**Convert segmentation dataset to yolo formated detection set**

In [ ]:
import os
import numpy as np
from pathlib import Path

def convert_segmentation_to_detection(label_content):
    """Convert segmentation format to detection format"""
    values = label_content.strip().split()
    if len(values) < 5:
        return None

    class_id = values[0]

    # Extract x,y coordinates from segmentation format
    coordinates = [float(x) for x in values[1:]]
    x_coords = coordinates[::2]
    y_coords = coordinates[1::2]

    # Calculate bounding box
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)

    # Convert to YOLO detection format (center_x, center_y, width, height)
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min

    # Return YOLO detection format string
    return f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"

def convert_dataset(base_path):
    """Convert entire dataset to detection format"""
    base_path = Path(base_path)

    for split in ['train', 'val', 'test']:
        labels_dir = base_path / split / 'labels'
        if not labels_dir.exists():
            continue

        print(f"Converting {split} labels...")
        for label_file in labels_dir.glob('*.txt'):
            try:
                # Read original content
                with open(label_file, 'r') as f:
                    content = f.read().strip()

                # Convert format
                new_content = convert_segmentation_to_detection(content)

                if new_content:
                    # Save in detection format
                    with open(label_file, 'w') as f:
                        f.write(new_content)

            except Exception as e:
                print(f"Error processing {label_file}: {str(e)}")

    print("Conversion complete!")

# Convert the dataset
dataset_path = '/content/YoloDataset'
convert_dataset(dataset_path)



**Creating synthetic data for multiple food items in one image**

In [ ]:
import cv2
import numpy as np
import random
import os
from pathlib import Path
import glob

def create_and_verify_synthetic_dataset():
    # Set paths
    dataset_path = '/content/YoloDatasetUpdated'
    synthetic_path = '/content/YoloDatasetUpdated/synthetic'

    # Create synthetic images
    print("Creating synthetic images...")
    create_synthetic_images(dataset_path, synthetic_path, num_synthetic=500)

    # Verify creation using glob
    synthetic_images = len(glob.glob(os.path.join(synthetic_path, 'images', '*.jpg')))
    synthetic_labels = len(glob.glob(os.path.join(synthetic_path, 'labels', '*.txt')))

    print(f"\nCreated synthetic dataset:")
    print(f"Images: {synthetic_images}")
    print(f"Labels: {synthetic_labels}")

    return synthetic_path

def resize_with_aspect_ratio(image, target_size=640):
    """Resize image while maintaining aspect ratio"""
    h, w = image.shape[:2]
    scale = min(target_size/w, target_size/h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    return cv2.resize(image, (new_w, new_h))

def create_synthetic_images(dataset_path, output_path, num_synthetic=1000):
    """Create synthetic images with multiple food items"""

    # Convert paths to strings and use glob
    images_path = os.path.join(dataset_path, 'train', 'images')
    labels_path = os.path.join(dataset_path, 'train', 'labels')
    output_images = os.path.join(output_path, 'images')
    output_labels = os.path.join(output_path, 'labels')

    # Create output directories
    os.makedirs(output_images, exist_ok=True)
    os.makedirs(output_labels, exist_ok=True)

    # Get list of image files using glob
    image_files = glob.glob(os.path.join(images_path, '*.jpg'))

    for i in range(num_synthetic):
        # Create blank canvas (white background)
        canvas = np.ones((640, 640, 3), dtype=np.uint8) * 255
        combined_labels = []

        # Randomly select 2-4 images to combine
        num_items = random.randint(2, 4)
        selected_images = random.sample(image_files, num_items)

        for idx, img_path in enumerate(selected_images):
            try:
                # Read image and label
                img = cv2.imread(img_path)
                if img is None:
                    continue

                # Get corresponding label path
                label_path = os.path.join(labels_path,
                                        os.path.splitext(os.path.basename(img_path))[0] + '.txt')
                if not os.path.exists(label_path):
                    continue

                with open(label_path, 'r') as f:
                    label = f.read().strip().split()

                # First resize to maintain aspect ratio
                img_resized = resize_with_aspect_ratio(img, target_size=320)
                new_h, new_w = img_resized.shape[:2]

                # Calculate random position
                x_pos = random.randint(0, max(0, 640 - new_w))
                y_pos = random.randint(0, max(0, 640 - new_h))

                # Create mask for smooth blending
                mask = np.ones(img_resized.shape[:2], dtype=np.float32)
                mask = cv2.GaussianBlur(mask, (7, 7), 0)

                # Place image on canvas with blending
                for c in range(3):
                    canvas[y_pos:y_pos+new_h, x_pos:x_pos+new_w, c] = \
                        canvas[y_pos:y_pos+new_h, x_pos:x_pos+new_w, c] * (1 - mask) + \
                        img_resized[:, :, c] * mask

                # Adjust label coordinates
                class_id = label[0]
                x_center = (x_pos + new_w/2) / 640
                y_center = (y_pos + new_h/2) / 640
                width = new_w / 640
                height = new_h / 640

                # Ensure coordinates are within bounds
                x_center = min(max(x_center, 0), 1)
                y_center = min(max(y_center, 0), 1)
                width = min(width, 1)
                height = min(height, 1)

                combined_labels.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

            except Exception as e:
                print(f"Error processing {img_path}: {str(e)}")
                continue

        if combined_labels:  # Only save if we have valid labels
            # Save synthetic image and labels
            output_img_path = os.path.join(output_images, f"synthetic_{i:04d}.jpg")
            output_label_path = os.path.join(output_labels, f"synthetic_{i:04d}.txt")

            cv2.imwrite(output_img_path, canvas)
            with open(output_label_path, 'w') as f:
                f.write('\n'.join(combined_labels))

            if i % 100 == 0:
                print(f"Created {i} synthetic images")

# Create the synthetic dataset
synthetic_path = create_and_verify_synthetic_dataset()

**Converting dataset with annotation.json to yolo compatible with image rotation**

In [2]:
import os
import json
import shutil
from PIL import Image
import math

def rotate_image_if_portrait(img):
    """
    Rotate image by -90 degrees if it's in portrait mode
    Returns rotated image and whether rotation was performed
    """
    width, height = img.size
    if height > width:
        return img.rotate(-90, expand=True), True
    return img, False

def convert_points_to_yolo(bb_coords, img_width, img_height):
    """
    Convert bounding box coordinates to YOLO format
    """
    x_min = min(bb_coords[0], bb_coords[4])
    x_max = max(bb_coords[0], bb_coords[4])
    y_min = min(bb_coords[1], bb_coords[5])
    y_max = max(bb_coords[1], bb_coords[5])

    width = (x_max - x_min) / float(img_width)
    height = (y_max - y_min) / float(img_height)
    center_x = (x_min + x_max) / (2.0 * img_width)
    center_y = (y_min + y_max) / (2.0 * img_height)

    width = min(max(width, 0.001), 1.0)
    height = min(max(height, 0.001), 1.0)
    center_x = min(max(center_x, 0.0), 1.0)
    center_y = min(max(center_y, 0.0), 1.0)

    return [center_x, center_y, width, height]

def process_dataset(input_dir, output_dir, class_mapping=None):
    """
    Convert dataset to YOLO format with image rotation
    """
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels'), exist_ok=True)

    if class_mapping is None:
        class_mapping = {}

    annotation_file = os.path.join(input_dir, 'annotation.json')
    with open(annotation_file, 'r') as f:
        annotations = json.load(f)

    print(f"Processing {len(annotations)} images in {input_dir}")

    class_counts = {}
    rotated_count = 0

    for img_name, objects in annotations.items():
        img_path = os.path.join(input_dir, f"{img_name}.jpg")

        try:
            # Load the image
            img = Image.open(img_path)

            # Rotate if needed
            rotated_img, was_rotated = rotate_image_if_portrait(img)
            if was_rotated:
                rotated_count += 1

            img_width, img_height = rotated_img.size

            yolo_annotations = []

            for obj in objects:
                for class_name, coords in obj.items():
                    if class_name not in class_mapping:
                        class_mapping[class_name] = len(class_mapping)

                    class_counts[class_name] = class_counts.get(class_name, 0) + 1

                    try:
                        bb_coords = coords['BB']
                        yolo_coords = convert_points_to_yolo(bb_coords, img_width, img_height)

                        class_id = class_mapping[class_name]
                        yolo_line = f"{class_id} {' '.join([f'{x:.6f}' for x in yolo_coords])}"
                        yolo_annotations.append(yolo_line)

                    except Exception as e:
                        print(f"Error processing annotation for {img_name}, class {class_name}: {str(e)}")
                        continue

            # Save rotated image and annotations
            if yolo_annotations:
                output_img_path = os.path.join(output_dir, 'images', f"{img_name}.jpg")
                rotated_img.save(output_img_path, quality=95)

                label_path = os.path.join(output_dir, 'labels', f"{img_name}.txt")
                with open(label_path, 'w') as f:
                    f.write('\n'.join(yolo_annotations))

            # Close images
            img.close()

        except Exception as e:
            print(f"Error processing image {img_name}: {str(e)}")
            continue

    print(f"\nRotated {rotated_count} portrait images to landscape")
    print("\nClass distribution:")
    for class_name, count in class_counts.items():
        print(f"{class_name}: {count}")

    return class_mapping

def create_dataset_yaml(output_dir, class_mapping):
    yaml_content = f"""
path: {os.path.abspath(output_dir)}
train: train/images
val: val/images
test: test/images

nc: {len(class_mapping)}
names: {list(class_mapping.keys())}

"""

    with open(os.path.join(output_dir, 'dataset.yaml'), 'w') as f:
        f.write(yaml_content)

def main():
    base_dir = '/content/newdataset/food'
    output_base_dir = '/content/YoloNewDataset'

    os.makedirs(output_base_dir, exist_ok=True)

    splits = ['train', 'val', 'test']
    class_mapping = None

    for split in splits:
        print(f"\nProcessing {split} split...")
        input_dir = os.path.join(base_dir, split)
        output_dir = os.path.join(output_base_dir, split)

        class_mapping = process_dataset(input_dir, output_dir, class_mapping)

    create_dataset_yaml(output_base_dir, class_mapping)

    with open(os.path.join(output_base_dir, 'class_mapping.json'), 'w') as f:
        json.dump(class_mapping, f, indent=2)

    print("\nConversion completed!")
    print(f"Class mapping: {class_mapping}")

if __name__ == "__main__":
    main()


Processing train split...
Processing 722 images in /content/newdataset/food/train

Rotated 0 portrait images to landscape

Class distribution:
pane: 370
pasta: 426
scaloppine: 76
carote: 130
yogurt: 94
pizza: 68
cotoletta: 99
mandarini: 150
patate/pure: 118
fagiolini: 98
spinaci: 84
budino: 85

Processing val split...
Processing 192 images in /content/newdataset/food/val

Rotated 0 portrait images to landscape

Class distribution:
pane: 84
pasta: 107
patate/pure: 25
fagiolini: 26
spinaci: 23
mandarini: 32
cotoletta: 42
budino: 20
scaloppine: 16
carote: 29
yogurt: 30
pizza: 17

Processing test split...
Processing 48 images in /content/newdataset/food/test

Rotated 0 portrait images to landscape

Class distribution:
mandarini: 16
pasta: 33
cotoletta: 7
fagiolini: 7
pane: 25
yogurt: 6
budino: 7
pizza: 4
spinaci: 3
scaloppine: 1
patate/pure: 8
carote: 2

Conversion completed!
Class mapping: {'pane': 0, 'pasta': 1, 'scaloppine': 2, 'carote': 3, 'yogurt': 4, 'pizza': 5, 'cotoletta': 6, 'man

**Visualise training data**

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

def validate_dataset(dataset_path):
    # Load class names from data.yaml
    import yaml
    with open(os.path.join(dataset_path, 'dataset.yaml'), 'r') as f:
        data_yaml = yaml.safe_load(f)
        class_names = data_yaml['names']

    images_path = os.path.join(dataset_path, 'train/images')
    labels_path = os.path.join(dataset_path, 'train/labels')

    validation_dir = 'label_validation'
    os.makedirs(validation_dir, exist_ok=True)

    # Adjustable font parameters
    FONT_SCALE = 1.2       # Increased font size
    FONT_THICKNESS = 2    # Thicker font
    LABEL_PADDING = 15     # More padding around label

    for img_file in os.listdir(images_path):
        img_path = os.path.join(images_path, img_file)
        label_file = os.path.join(labels_path, img_file.rsplit('.', 1)[0] + '.txt')

        img = cv2.imread(img_path)
        height, width = img.shape[:2]

        with open(label_file, 'r') as f:
            labels = f.readlines()

        for label in labels:
            try:
                class_id, x_center, y_center, w, h = map(float, label.strip().split())

                # Convert YOLO format to pixel coordinates
                x1 = int((x_center - w/2) * width)
                y1 = int((y_center - h/2) * height)
                x2 = int((x_center + w/2) * width)
                y2 = int((y_center + h/2) * height)

                # Draw box
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)  # Thicker box

                # Add class label text
                class_name = class_names[int(class_id)]
                label_text = f"{class_name} ({class_id})"

                # Get text size with new font parameters
                text_size = cv2.getTextSize(
                    label_text,
                    cv2.FONT_HERSHEY_SIMPLEX,
                    FONT_SCALE,
                    FONT_THICKNESS
                )[0]

                # Draw label background with padding
                cv2.rectangle(
                    img,
                    (x1, y1 - text_size[1] - LABEL_PADDING),
                    (x1 + text_size[0] + LABEL_PADDING//2, y1),
                    (0, 255, 0),
                    -1
                )

                # Draw label text
                cv2.putText(
                    img,
                    label_text,
                    (x1 + LABEL_PADDING//4, y1 - LABEL_PADDING//2),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    FONT_SCALE,
                    (0, 0, 0),
                    FONT_THICKNESS
                )

                print(f"\nImage: {img_file}")
                print(f"Class: {class_name} (ID: {class_id})")
                print(f"YOLO format: {x_center:.4f}, {y_center:.4f}, {w:.4f}, {h:.4f}")
                print(f"Pixel coordinates: ({x1}, {y1}), ({x2}, {y2})")

            except Exception as e:
                print(f"Error processing label in {img_file}: {str(e)}")

        output_path = os.path.join(validation_dir, f'validation_{img_file}')
        cv2.imwrite(output_path, img)

# Use the function
dataset_path = '/content/YoloNewDataset'
validate_dataset(dataset_path)

**Label analysis**

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

def analyze_current_labels(dataset_path):
    """Analyze and visualize current label coordinates"""
    images_dir = os.path.join(dataset_path, 'train/images')
    labels_dir = os.path.join(dataset_path, 'train/labels')

    # Analysis statistics
    coordinate_stats = {
        'x_centers': [],
        'y_centers': [],
        'widths': [],
        'heights': []
    }

    for img_name in os.listdir(images_dir):
        base_name = os.path.splitext(img_name)[0]
        img_path = os.path.join(images_dir, img_name)
        label_path = os.path.join(labels_dir, f"{base_name}.txt")

        if not os.path.exists(label_path):
            continue

        # Read image
        img = cv2.imread(img_path)
        height, width = img.shape[:2]

        # Read label
        with open(label_path, 'r') as f:
            labels = f.readlines()

        for label in labels:
            try:
                class_id, x_center, y_center, w, h = map(float, label.strip().split())

                # Store normalized coordinates
                coordinate_stats['x_centers'].append(x_center)
                coordinate_stats['y_centers'].append(y_center)
                coordinate_stats['widths'].append(w)
                coordinate_stats['heights'].append(h)

                # Convert to pixel coordinates for visualization
                x1 = int((x_center - w/2) * width)
                y1 = int((y_center - h/2) * height)
                x2 = int((x_center + w/2) * width)
                y2 = int((y_center + h/2) * height)

                # Draw current box in red
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)

                # Draw center point
                center_x = int(x_center * width)
                center_y = int(y_center * height)
                cv2.circle(img, (center_x, center_y), 5, (255, 0, 0), -1)

                print(f"\nImage: {img_name}")
                print(f"Normalized coordinates: x={x_center:.3f}, y={y_center:.3f}, w={w:.3f}, h={h:.3f}")
                print(f"Pixel coordinates: ({x1}, {y1}) to ({x2}, {y2})")

            except Exception as e:
                print(f"Error processing {img_name}: {str(e)}")

        # Save visualization
        output_dir = 'label_analysis'
        os.makedirs(output_dir, exist_ok=True)
        cv2.imwrite(os.path.join(output_dir, f'analysis_{img_name}'), img)

    # Plot coordinate distributions
    plt.figure(figsize=(15, 10))

    plt.subplot(221)
    plt.hist(coordinate_stats['x_centers'], bins=50)
    plt.title('X Center Distribution')

    plt.subplot(222)
    plt.hist(coordinate_stats['y_centers'], bins=50)
    plt.title('Y Center Distribution')

    plt.subplot(223)
    plt.hist(coordinate_stats['widths'], bins=50)
    plt.title('Width Distribution')

    plt.subplot(224)
    plt.hist(coordinate_stats['heights'], bins=50)
    plt.title('Height Distribution')

    plt.tight_layout()
    plt.savefig('coordinate_distributions.png')
    plt.close()

    return coordinate_stats

# Run analysis
dataset_path = '/content/YoloNewDataset'
stats = analyze_current_labels(dataset_path)

# Print summary statistics
print("\nCoordinate Statistics:")
for key, values in stats.items():
    print(f"\n{key}:")
    print(f"Min: {min(values):.3f}")
    print(f"Max: {max(values):.3f}")
    print(f"Mean: {np.mean(values):.3f}")
    print(f"Std: {np.std(values):.3f}")

**Train combined dataset**

In [11]:
from ultralytics import YOLO

def train_combined_dataset():
    # Start with pretrained model
    model = YOLO('yolo11n.pt')

    # Train with combined data
    results = model.train(
        data='/content/YoloNewDataset/dataset.yaml',
        #data='/content/FoodObjectDetectionYolov11/data.yaml',
        epochs=100,
        imgsz=640,
        batch=16,
        patience=20,
        lr0=0.01,
        # Multiple object detection parameters
        mosaic=1.0,
        mixup=0.5,
        copy_paste=0.5,
        max_det=20
    )
    return model

model = train_combined_dataset()

Ultralytics 8.3.53 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=/content/YoloNewDataset/dataset.yaml, epochs=100, time=None, patience=20, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=20, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, 

train: Scanning /content/YoloNewDataset/train/labels.cache... 722 images, 0 backgrounds, 0 corrupt: 100%|██████████| 722/722 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))



val: Scanning /content/YoloNewDataset/val/labels.cache... 192 images, 0 backgrounds, 0 corrupt: 100%|██████████| 192/192 [00:00<?, ?it/s]


Plotting labels to runs/detect/train2/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000625, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to runs/detect/train2
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

     95/100      2.47G     0.2737     0.2533     0.8009          4        640: 100%|██████████| 46/46 [02:36<00:00,  3.40s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:23<00:00,  3.85s/it]

                   all        192        451      0.973      0.953      0.972      0.907
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 75, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



95 epochs completed in 5.600 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 5.5MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 5.5MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.53 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
YOLO11n summary (fused): 238 layers, 2,584,492 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:30<00:00,  5.15s/it]


                   all        192        451      0.976       0.95      0.983       0.92
                  pane         77         84      0.988          1      0.995      0.964
                 pasta        107        107      0.998          1      0.995      0.983
            scaloppine         16         16      0.994      0.812      0.969      0.834
                carote         29         29          1      0.996      0.995      0.916
                yogurt         30         30      0.935          1      0.986      0.923
                 pizza         16         17      0.984      0.941       0.95      0.924
             cotoletta         42         42          1          1      0.995      0.973
             mandarini         15         32      0.956      0.812      0.953      0.881
           patate/pure         25         25          1      0.888      0.992      0.867
             fagiolini         26         26      0.927          1      0.994       0.93
               spinac

In [28]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"
import time
import shutil
timestr = time.strftime("%Y%m%d-%H%M%S")
print(timestr)

modelName = f"trained_model_{timestr}.pt"
print(modelName)
shutil.copy("/content/runs/detect/train2/weights/best.pt", modelName)

20241222-221022
trained_model_20241222-221022.pt


'trained_model_20241222-221022.pt'

**Inference the model**

In [17]:
from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image

def run_inference():
    # Load your trained model
    model = YOLO('/content/trained_model.pt')  # adjust path as needed

    # Run inference on an image
    results = model.predict(
        source='/content/YoloNewDataset/val/images/20151127_121613.jpg',  # can be image/folder/video
        save=True,                # save results
        conf=0.1,               # confidence threshold
        iou=0.45,
        save_txt=True,           # save results in txt file
        save_crop=True           # save cropped predictions
    )

    # Process results
    for result in results:
        # Get masks
        if result.masks is not None:
            masks = result.masks.data.cpu().numpy()

        # Get boxes
        if result.boxes is not None:
            boxes = result.boxes.data.cpu().numpy()

        # Get class names for predictions
        class_names = [model.names[int(class_id)] for class_id in result.boxes.cls]

        print(f"Found {len(class_names)} objects: {class_names}")


run_inference()




image 1/1 /content/YoloNewDataset/val/images/20151127_121613.jpg: 480x640 1 pane, 1 pasta, 1 scaloppine, 2 mandarinis, 1 spinaci, 10.9ms
Speed: 3.2ms preprocess, 10.9ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)
Results saved to runs/detect/predict4
1 label saved to runs/detect/predict4/labels
Found 6 objects: ['pane', 'mandarini', 'mandarini', 'pasta', 'spinaci', 'scaloppine']


**Validate model**

In [18]:
from ultralytics import YOLO

# Load model
model = YOLO('/content/runs/detect/train2/weights/best.pt')

# Run model validation on the validation set
metrics = model.val(data='/content/YoloNewDataset/dataset.yaml')
print("Validation Metrics:", metrics)

Ultralytics 8.3.53 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
YOLO11n summary (fused): 238 layers, 2,584,492 parameters, 0 gradients, 6.3 GFLOPs


val: Scanning /content/YoloNewDataset/val/labels.cache... 192 images, 0 backgrounds, 0 corrupt: 100%|██████████| 192/192 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:41<00:00,  3.48s/it]


                   all        192        451      0.975       0.95      0.983       0.92
                  pane         77         84      0.988          1      0.995      0.962
                 pasta        107        107      0.998          1      0.995      0.983
            scaloppine         16         16      0.993      0.812      0.969      0.839
                carote         29         29          1      0.997      0.995      0.914
                yogurt         30         30      0.935          1      0.985      0.919
                 pizza         16         17      0.984      0.941       0.95      0.923
             cotoletta         42         42          1          1      0.995      0.972
             mandarini         15         32      0.955      0.812      0.953      0.881
           patate/pure         25         25          1      0.889      0.992      0.867
             fagiolini         26         26      0.927          1      0.994      0.935
               spinac

In [19]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"
!zip -r YoloDatasetUnimebRuns.zip /content/runs

  adding: content/runs/ (stored 0%)
  adding: content/runs/detect/ (stored 0%)
  adding: content/runs/detect/train/ (stored 0%)
  adding: content/runs/detect/train/args.yaml (deflated 53%)
  adding: content/runs/detect/train/train_batch690.jpg (deflated 7%)
  adding: content/runs/detect/train/val_batch1_pred.jpg (deflated 6%)
  adding: content/runs/detect/train/labels_correlogram.jpg (deflated 43%)
  adding: content/runs/detect/train/PR_curve.png (deflated 16%)
  adding: content/runs/detect/train/results.csv (deflated 60%)
  adding: content/runs/detect/train/R_curve.png (deflated 10%)
  adding: content/runs/detect/train/train_batch1.jpg (deflated 2%)
  adding: content/runs/detect/train/labels.jpg (deflated 29%)
  adding: content/runs/detect/train/results.png (deflated 8%)
  adding: content/runs/detect/train/train_batch691.jpg (deflated 8%)
  adding: content/runs/detect/train/P_curve.png (deflated 8%)
  adding: content/runs/detect/train/events.out.tfevents.1734880553.d7c5edfbe989.855.0 